# 04 · 取樣：DDPM vs DDIM，與去噪軌跡

上一課用 `ddpm_sample` 生成,它老老實實走完全部 T=200 步,**慢**。這課認識更快的取樣法,並把「雪花 → 數字」的過程**逐步畫出來**。

> **DDPM**:標準反向過程,每步都加一點隨機性,要走完全部 T 步。
> **DDIM**:把取樣變成**確定性**的,而且可以**跳著走**——只要 20 步就能生出差不多的圖,快 10 倍。

## 1. 設定、模型、排程、資料(同前)

In [ ]:
%pip install -q -U torchvision

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class SinusoidalPosEmb(nn.Module):
    """把整數時間步 t 編碼成向量(同 Transformer 的位置編碼)。"""

    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device) / (half - 1)
        )
        args = t[:, None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)


class ResBlock(nn.Module):
    """conv 殘差塊,並把時間嵌入加進特徵圖。"""

    def __init__(self, cin, cout, tdim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, cin)
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.temb = nn.Linear(tdim, cout)
        self.norm2 = nn.GroupNorm(8, cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, temb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.temb(temb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class TinyUNet(nn.Module):
    """最小可用的 U-Net:下採樣兩次、上採樣兩次,帶 skip 連接。"""

    def __init__(self, base=32, tdim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(tdim), nn.Linear(tdim, tdim), nn.SiLU(), nn.Linear(tdim, tdim)
        )
        self.in_conv = nn.Conv2d(1, base, 3, padding=1)
        self.d1 = ResBlock(base, base, tdim)
        self.down1 = nn.Conv2d(base, base, 4, 2, 1)
        self.d2 = ResBlock(base, base * 2, tdim)
        self.down2 = nn.Conv2d(base * 2, base * 2, 4, 2, 1)
        self.mid = ResBlock(base * 2, base * 2, tdim)
        self.up2 = nn.ConvTranspose2d(base * 2, base * 2, 4, 2, 1)
        self.u2 = ResBlock(base * 2 + base * 2, base, tdim)
        self.up1 = nn.ConvTranspose2d(base, base, 4, 2, 1)
        self.u1 = ResBlock(base + base, base, tdim)
        self.out = nn.Sequential(nn.GroupNorm(8, base), nn.SiLU(), nn.Conv2d(base, 1, 3, padding=1))

    def forward(self, x, t):
        temb = self.time_mlp(t)
        x = self.in_conv(x)
        s1 = self.d1(x, temb)
        x = self.down1(s1)
        s2 = self.d2(x, temb)
        x = self.down2(s2)
        x = self.mid(x, temb)
        x = self.up2(x)
        x = self.u2(torch.cat([x, s2], dim=1), temb)
        x = self.up1(x)
        x = self.u1(torch.cat([x, s1], dim=1), temb)
        return self.out(x)

In [ ]:
import torch

T = 200                                                  # 擴散步數
betas = torch.linspace(1e-4, 0.02, T, device=device)     # 線性 beta 排程
alphas = 1.0 - betas
alphabars = torch.cumprod(alphas, dim=0)                  # 連乘:一步到位的關鍵


def q_sample(x0, t, noise):
    """前向加噪(closed form):x_t = sqrt(ab)·x0 + sqrt(1-ab)·noise。"""
    ab = alphabars[t].view(-1, 1, 1, 1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * noise


@torch.no_grad()
def ddpm_sample(model, n=16):
    """反向去噪:從純噪聲一步步還原,共 T 步(慢但標準)。"""
    model.eval()
    x = torch.randn(n, 1, 32, 32, device=device)
    for i in reversed(range(T)):
        t = torch.full((n,), i, device=device, dtype=torch.long)
        eps = model(x, t)
        a, ab, beta = alphas[i], alphabars[i], betas[i]
        mean = (x - (1 - a) / (1 - ab).sqrt() * eps) / a.sqrt()
        x = mean + beta.sqrt() * torch.randn_like(x) if i > 0 else mean
    return x


@torch.no_grad()
def ddim_sample(model, n=16, steps=20):
    """DDIM:確定性取樣,跳著走、只要 steps 步(快很多)。"""
    model.eval()
    x = torch.randn(n, 1, 32, 32, device=device)
    seq = torch.linspace(T - 1, 0, steps, device=device).long().tolist()
    for j, i in enumerate(seq):
        t = torch.full((n,), i, device=device, dtype=torch.long)
        eps = model(x, t)
        ab = alphabars[i]
        x0 = (x - (1 - ab).sqrt() * eps) / ab.sqrt()         # 預測乾淨影像
        if j < len(seq) - 1:
            ab_next = alphabars[seq[j + 1]]
            x = ab_next.sqrt() * x0 + (1 - ab_next).sqrt() * eps
        else:
            x = x0
    return x

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

tf = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),     # → [-1, 1]
])
mnist = torchvision.datasets.MNIST("./data", train=True, download=True, transform=tf)
loader = DataLoader(mnist, batch_size=128, shuffle=True, num_workers=2)
print(f"MNIST {len(mnist)} 張,縮到 32×32、正規化到 [-1,1]")

In [ ]:
import torch
import torch.nn.functional as F


def train_diffusion(model, loader, epochs=5, lr=2e-4):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(1, epochs + 1):
        model.train()
        total, nb = 0.0, 0
        for x, _ in loader:
            x = x.to(device)
            t = torch.randint(0, T, (x.size(0),), device=device)
            noise = torch.randn_like(x)
            pred = model(q_sample(x, t, noise), t)        # 預測加進去的噪聲
            loss = F.mse_loss(pred, noise)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); nb += 1
        print(f"epoch {ep}/{epochs}  loss {total / nb:.4f}")
    return model

## 2. 快速訓練一個去噪器

為了這課自成一體,先訓練幾個 epoch(同上一課)。

In [ ]:
model = train_diffusion(model := TinyUNet().to(device), loader, epochs=5)

## 3. DDPM(200 步)vs DDIM(20 步)

同一個模型,兩種取樣。DDIM 用十分之一的步數,品質卻接近。

In [ ]:
import matplotlib.pyplot as plt


def show_gen(imgs, cols=8, title=None):
    imgs = ((imgs.clamp(-1, 1) + 1) / 2).detach().cpu()    # [-1,1] → [0,1]
    n = len(imgs); rows = (n + cols - 1) // cols
    plt.figure(figsize=(cols * 1.1, rows * 1.15))
    for i in range(n):
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(imgs[i, 0], cmap="gray"); ax.axis("off")
    if title:
        plt.suptitle(title)
    plt.tight_layout(); plt.show()

In [ ]:
import time

t0 = time.time(); ddpm = ddpm_sample(model, n=8); t_ddpm = time.time() - t0
t0 = time.time(); ddim = ddim_sample(model, n=8, steps=20); t_ddim = time.time() - t0
print(f"DDPM 200 步:{t_ddpm:.1f}s")
print(f"DDIM  20 步:{t_ddim:.1f}s  (快約 {t_ddpm / max(t_ddim, 1e-6):.0f} 倍)")
show_gen(ddpm, cols=8, title="DDPM (200 steps)")
show_gen(ddim, cols=8, title="DDIM (20 steps)")

## 4. 把去噪軌跡畫出來

從純噪聲到清晰數字,中間長什麼樣?抓 DDIM 過程中的幾個快照,看它一步步「顯影」。

In [ ]:
import torch
import matplotlib.pyplot as plt

@torch.no_grad()
def ddim_trajectory(model, steps=20, snaps=6):
    model.eval()
    x = torch.randn(1, 1, 32, 32, device=device)
    seq = torch.linspace(T - 1, 0, steps, device=device).long().tolist()
    frames = []
    for j, i in enumerate(seq):
        t = torch.full((1,), i, device=device, dtype=torch.long)
        eps = model(x, t); ab = alphabars[i]
        x0 = (x - (1 - ab).sqrt() * eps) / ab.sqrt()
        x = (alphabars[seq[j + 1]].sqrt() * x0 + (1 - alphabars[seq[j + 1]]).sqrt() * eps
             if j < len(seq) - 1 else x0)
        frames.append(x.clone())
    idx = [int(k) for k in torch.linspace(0, len(frames) - 1, snaps)]
    fig, axes = plt.subplots(1, snaps, figsize=(11, 2.2))
    for ax, k in zip(axes, idx):
        ax.imshow(((frames[k][0, 0].clamp(-1, 1) + 1) / 2).cpu(), cmap="gray")
        ax.set_title(f"step {k}", fontsize=10); ax.axis("off")
    plt.tight_layout(); plt.show()

ddim_trajectory(model)

## 小結

- **DDPM**(全 T 步、有隨機性)vs **DDIM**(可跳步、確定性):後者快一個量級,品質接近。
- 真實的 Stable Diffusion 也是用 DDIM 這類**少步快取樣**,你才不用等好幾分鐘。
- 去噪軌跡顯示生成是**逐步顯影**:從雪花裡慢慢「凝結」出結構。

下一課:跨入**文字條件**——怎麼讓「一句話」導引生成什麼圖(CLIP 與 text embedding)。